In [ ]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 0.13
# ===================================================================
# AUTOR: Carmen Gómez Alarcón
# FECHA: 2026
# FASE: Paso 2. Análisis de bloques de ocupación
# DESCRIPCIÓN:
#   Extrae y analiza bloques continuos de ocupación a partir de datos
#   semanales de sensores (step 1). Identifica períodos con niveles constantes
#   de ocupación (0, 10, 30, etc.) y calcula estadísticas de otras
#   variables ambientales (temperatura, humedad, CO2, etc.) dentro
#   de cada bloque. Genera un archivo Excel con todos los bloques
#   identificados, incluyendo duración, horarios y estadísticas de
#   sensores asociados.
# ===================================================================

import pandas as pd
import numpy as np
import os
from openpyxl.styles import PatternFill, Font
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# CONFIGURACIÓN
# ==========================================
INPUT_DIR  = '.'   # Directorio donde están los archivos Excel semanales
OUTPUT_DIR = 'occupancy_analysis'  # Directorio para guardar los resultados
os.makedirs(OUTPUT_DIR, exist_ok=True)  # Crear directorio si no existe

# Columnas de metadatos que identifican características de las mediciones
metadata_cols = ['Room', 'Volume', 'Floor', 'Sensor_Location', 'Variable_Measure']

def extract_occupancy_blocks(df_week, week_name):
    
    # Buscar la fila que contiene la variable 'Occupancy'
    occ_row = df_week[df_week['Variable_Measure'] == 'Occupancy']
    if occ_row.empty:
        return []  # No hay datos de ocupación

    occ_row = occ_row.iloc[0]  # Tomar la primera fila de ocupación
    # Obtener columnas de fechas (excluyendo metadatos)
    date_cols = sorted([c for c in df_week.columns if c not in metadata_cols])

    # Extraer la secuencia temporal de ocupación
    occupancy_sequence = []
    for col in date_cols:
        val = occ_row[col]
        if pd.notna(val):  # Solo si hay valor no nulo
            try:
                occupancy_sequence.append({
                    'DateTime': pd.to_datetime(col),  # Convertir a datetime
                    'Occupancy': int(val)  # Valor de ocupación (0, 10, 30, etc)
                })
            except:
                continue  # Ignorar errores de conversión

    if not occupancy_sequence:
        return []

    # Identificar bloques contiguos del mismo nivel de ocupación
    blocks = []
    current_block = [occupancy_sequence[0]]  # Iniciar primer bloque
    for i in range(1, len(occupancy_sequence)):
        # Si cambia el nivel de ocupación, cerrar bloque actual y empezar nuevo
        if occupancy_sequence[i]['Occupancy'] != occupancy_sequence[i-1]['Occupancy']:
            blocks.append(current_block)
            current_block = [occupancy_sequence[i]]
        else:
            current_block.append(occupancy_sequence[i])  # Mismo nivel, continuar bloque
    blocks.append(current_block)  # Agregar último bloque

    # Filtrar sensores (excluir ocupación, puerta y ventana)
    sensor_data = df_week[~df_week['Variable_Measure'].isin(['Occupancy', 'Door_Open', 'Window_Open'])]
    date_cols_dt = [pd.to_datetime(c) for c in date_cols]  # Convertir fechas una sola vez

    rows = []
    # Procesar cada bloque de ocupación
    for block in blocks:
        block_start = block[0]['DateTime']  # Inicio del bloque
        block_end   = block[-1]['DateTime']  # Fin del bloque
        # Encontrar índices de columnas que caen dentro del bloque
        block_indices = [i for i, dt in enumerate(date_cols_dt) if block_start <= dt <= block_end]

        if not block_indices:
            continue  # Saltar si no hay índices

        # Información básica del bloque de ocupación
        row = {
            'Occupancy':    block[0]['Occupancy'],  # Nivel de ocupación
            'Date':         block_start.date(),  # Fecha del bloque
            'Day_of_Week':  block_start.strftime('%A'),  # Día de la semana
            'Start_Time':   block_start.time(),  # Hora de inicio
            'End_Time':     block_end.time(),  # Hora de fin
            'Duration_min': round((block_end - block_start).total_seconds() / 60),  # Duración en minutos
            'Week':         week_name  # Identificador de la semana
        }

        # Calcular estadísticas para cada sensor en el bloque
        for _, sensor_row in sensor_data.iterrows():
            # Extraer valores del sensor dentro del bloque temporal
            values = [float(sensor_row[date_cols[idx]]) for idx in block_indices
                      if pd.notna(sensor_row[date_cols[idx]])]
            if values:
                # Construir nombre de columna con ubicación y variable medida
                prefix = f"{sensor_row['Sensor_Location']}_{sensor_row['Variable_Measure']}" \
                         if sensor_row['Sensor_Location'] else sensor_row['Variable_Measure']
                # Agregar estadísticas descriptivas
                row.update({
                    f'{prefix}_mean': np.mean(values),  # Media
                    f'{prefix}_std':  np.std(values) if len(values) > 1 else 0,  # Desviación estándar
                    f'{prefix}_max':  np.max(values),  # Máximo
                    f'{prefix}_min':  np.min(values)   # Mínimo
                })
        rows.append(row)

    return rows

# ==========================================
# CARGAR EXCEL SEMANALES Y EXTRAER BLOQUES
# ==========================================

# Buscar todos los archivos semanales que comienzan con '013_data_'
weekly_files = sorted([f for f in os.listdir(INPUT_DIR) if f.startswith('013_data_') and f.endswith('.xlsx')])
print(f"Encontrados {len(weekly_files)} archivos semanales")

all_occupancy_rows = []  # Lista para acumular todos los bloques
# Procesar cada archivo semanal
for fname in weekly_files:
    print(f"   Procesando {fname}...")
    df_week = pd.read_excel(os.path.join(INPUT_DIR, fname))  # Cargar Excel
    week_name = fname.replace('013_data_', '').replace('.xlsx', '')  # Extraer nombre de semana
    rows = extract_occupancy_blocks(df_week, week_name)  # Extraer bloques
    all_occupancy_rows.extend(rows)  # Acumular resultados
    print(f"      Bloques encontrados: {len(rows)}")

# ==========================================
# GUARDAR RESULTADOS
# ==========================================

if all_occupancy_rows:
    # Crear DataFrame con todos los bloques
    df_occupancy = pd.DataFrame(all_occupancy_rows)
    
    # Reordenar columnas para mejor legibilidad
    first_cols = ['Occupancy', 'Date', 'Day_of_Week', 'Start_Time', 'End_Time', 'Duration_min', 'Week']
    other_cols = sorted([c for c in df_occupancy.columns if c not in first_cols])
    df_occupancy = df_occupancy[first_cols + other_cols].sort_values(['Date', 'Start_Time'])

    # Guardar a archivo Excel con formato
    file_final = os.path.join(OUTPUT_DIR, '013_occupancy_blocks_analysis.xlsx')
    with pd.ExcelWriter(file_final, engine='openpyxl') as writer:
        df_occupancy.to_excel(writer, sheet_name='Occupancy_Blocks', index=False)
        ws = writer.sheets['Occupancy_Blocks']
        
        # Formato de encabezados: fondo azul, texto blanco y negrita
        for cell in ws[1]:
            cell.fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
            cell.font = Font(bold=True, size=11, color='FFFFFF')
        
        # Aplicar colores según nivel de ocupación
        occ_col = list(df_occupancy.columns).index('Occupancy') + 1  # Columna de ocupación (1-indexed)
        # Diccionario: umbral -> color (colores según nivel)
        colors = {0: 'F2F2F2',  # Gris claro para ocupación 0
                 10: 'E2EFDA',  # Verde claro para 10
                 30: 'FCE4D6'}  # Naranja claro para 30
        for row in range(2, ws.max_row + 1):
            val = ws.cell(row=row, column=occ_col).value
            if val is not None:
                # Elegir color según umbral (default: rosa para valores >30)
                color = next((c for threshold, c in sorted(colors.items()) if val < threshold), 'F4B4C2')
                ws.cell(row=row, column=occ_col).fill = PatternFill(start_color=color, end_color=color, fill_type='solid')

    print(f"\nGuardado en: {file_final}")
    print(f"Total de bloques: {len(df_occupancy)}")
    print(f"Niveles de ocupación encontrados: {sorted(df_occupancy['Occupancy'].unique())}")
else:
    print("No se encontraron bloques de ocupación.")

print("\n¡Proceso completado!")